In [ ]:
import os
import sqlite3
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

# In notebooks, __file__ is not defined. Use the current working directory.
CWD = Path.cwd()
BASE_DIR = CWD.parent if CWD.name == "notebooks" else CWD
DB_PATH = BASE_DIR / "codecity.db"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Using DB:", DB_PATH)
print("Saving models to:", MODELS_DIR)

conn = sqlite3.connect(str(DB_PATH))
files_df = pd.read_sql_query(
    "SELECT size, complexity, width, depth, height, is_test_file, area, aspect_ratio FROM files",
    conn,
)
conn.close()

feature_cols = [
    "size",
    "complexity",
    "width",
    "depth",
    "height",
    "is_test_file",
    "area",
    "aspect_ratio",
]

X = files_df[feature_cols].values.astype(float)

iso = IsolationForest(
    n_estimators=200,
    contamination=0.1,
    random_state=42,
)
iso.fit(X)

anomaly_scores = -iso.score_samples(X)  # higher = more anomalous
files_df["anomaly_score"] = anomaly_scores

model_path = MODELS_DIR / "anomaly_model.joblib"
meta_path = MODELS_DIR / "anomaly_model_meta.json"

joblib.dump(iso, str(model_path))

with open(meta_path, "w") as f:
    json.dump({"feature_cols": feature_cols}, f, indent=2)

print("Saved anomaly model to", model_path)
print("Saved metadata to", meta_path)
display(files_df[["complexity", "anomaly_score"]].head())